<a href="https://colab.research.google.com/github/lcbjrrr/DBMS/blob/main/LevelDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
APIKEY=""

![](https://pbs.twimg.com/media/HQLZ1BaXEAAD7zl?format=jpg&name=medium)


**Transactional SQL**: Relational database systems prioritizing strict ACID compliance, fixed schemas, and complex multi-table joins within a single geographical location or region. Regional e-commerce ordering platforms, localized accounting systems, or enterprise HR applications where transactional integrity is critical, but traffic remains concentrated within one region. ***(MySQL / SQLite)***

*Global Scalability*- Worldwide core-banking networks, international flight reservation systems, or global ledger applications where double-spending or overbooking across continents must be strictly prevented. ***(Postgres)***


**Transactional NoSQL**: Dynamic, schema-flexible document or key-value stores engineered for high-concurrency, low-latency CRUD operations and seamless client-side data synchronization. Mobile application backends, real-time gaming state managers, active session stores, and dynamic user profile catalogs that require fast reads and writes without complex table relationships. ***(MongoDB)***


**Analytical SQL**: Columnar, massively parallel processing (MPP) data warehouses optimized for executing complex analytical queries, aggregations, and scanning vast historical datasets using standard SQL syntax. Enterprise business intelligence, cross-departmental reporting dashboards, customer churn analysis, and multi-year financial trend forecasting. ***(Iceberg / Databricks)***


**Analytical NoSQL**: High-throughput wide-column or key-value engines built to handle massive, continuous write streams and fast, sequential range scans on append-heavy data. Internet of Things (IoT) sensor telemetry, real-time user clickstream tracking, high-frequency financial ticker ingestion, and operational log monitoring. ***(LevelDB)***


In [ ]:
! apt-get update
! apt-get install -y build-essential libleveldb-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [110 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,167 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,230 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,315 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,623 kB]
Ge

In [ ]:
!pip install plyvel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.8/937.8 kB 7.5 MB/s eta 0:00:00


In [ ]:
!pip install finnhub-python

In [ ]:
import finnhub
finnhub_client = finnhub.Client(api_key=APIKEY)
finnhub_client.quote('BINANCE:DOGEUSDT')['c']

0.08031

In [ ]:
import sys
import subprocess
import time
import json
import finnhub
import plyvel
import os

# Initialize Finnhub client using the APIKEY available in the kernel
finnhub_client = finnhub.Client(api_key=APIKEY)

# Initialize LevelDB database
db = plyvel.DB('./LevelDB', create_if_missing=True)
ITS=111
TIM=4
print("Starting data collection and storage...")
for i in range(ITS):
    try:
        # Make the API call
        quote_data = finnhub_client.quote('BINANCE:DOGEUSDT')
        if quote_data and 't' in quote_data:
            # Extract 't' for the key
            key = str(quote_data['t']).encode('utf-8')
            # Create a dictionary for the value, excluding 't'
            value_data = {k: v for k, v in quote_data.items() if k != 't'}
            # Serialize the value dictionary to JSON string and encode to bytes
            value = json.dumps(value_data).encode('utf-8')
            # Insert into LevelDB using db.put()
            db.put(key, value)
            print(f"[{i+1}/{ITS}] Stored data for timestamp: {quote_data['t']}", value)
        else:
            print(f"[{i+1}/{ITS}] No 't' key found in quote data or quote data is empty. Skipping this iteration.")
    except Exception as e:
        print(f"[{i+1}/{ITS}] An error occurred during API call or LevelDB operation: {e}")
    # Sleep for 1 second before the next call
    time.sleep(TIM)
print("Data collection complete.")



Starting data collection and storage...
[1/111] Stored data for timestamp: 1787241521 b'{"c": 0.08045, "d": 0.00787, "dp": 10.8432, "h": 0.08066, "l": 0.07192, "o": 0.07258, "pc": 0.07258}'
[2/111] Stored data for timestamp: 1787241521 b'{"c": 0.08045, "d": 0.00787, "dp": 10.8432, "h": 0.08066, "l": 0.07192, "o": 0.07258, "pc": 0.07258}'
[3/111] Stored data for timestamp: 1787241521 b'{"c": 0.08045, "d": 0.00787, "dp": 10.8432, "h": 0.08066, "l": 0.07192, "o": 0.07258, "pc": 0.07258}'
[4/111] Stored data for timestamp: 1787241521 b'{"c": 0.08045, "d": 0.00787, "dp": 10.8432, "h": 0.08066, "l": 0.07192, "o": 0.07258, "pc": 0.07258}'
[5/111] Stored data for timestamp: 1787241540 b'{"c": 0.08033, "d": 0.00778, "dp": 10.7236, "h": 0.08066, "l": 0.07192, "o": 0.07254, "pc": 0.07255}'
[6/111] Stored data for timestamp: 1787241540 b'{"c": 0.08033, "d": 0.00778, "dp": 10.7236, "h": 0.08066, "l": 0.07192, "o": 0.07254, "pc": 0.07255}'
[7/111] Stored data for timestamp: 1787241540 b'{"c": 0.0803

In [ ]:
KEY=1787241521
start_time = time.perf_counter()
k = db.get(str(KEY).encode('utf-8'))
if k:
  data = json.loads(k.decode('utf-8'))
  print(f"Value for key {k}:\n{json.dumps(data, indent=2)}")
elapsed_time = time.perf_counter() - start_time
print(elapsed_time*1000,' ms')

Value for key b'{"c": 0.08045, "d": 0.00787, "dp": 10.8432, "h": 0.08066, "l": 0.07192, "o": 0.07258, "pc": 0.07258}':
{
  "c": 0.08045,
  "d": 0.00787,
  "dp": 10.8432,
  "h": 0.08066,
  "l": 0.07192,
  "o": 0.07258,
  "pc": 0.07258
}
0.43957999992016994  ms


In [ ]:
start_time = time.perf_counter()
TH=0.082
found_count=0
for key, value in db.iterator():
    try:
        timestamp = key.decode('utf-8')
        record = json.loads(value.decode('utf-8'))
        if 'c' in record and record['c'] < TH:
            #d=json.dumps(record, indent=2)
            print(f"Found entry with timestamp: {timestamp}",f"  Data: {record['c']}")
            found_count += 1
    except Exception as e:
        print(f"An error occurred processing entry with key {key}: {e}")
print("Entries found meeting the criteria:",found_count)
elapsed_time = time.perf_counter() - start_time
print(elapsed_time*1000,' ms')

Found entry with timestamp: 1787241521   Data: 0.08045
Found entry with timestamp: 1787241540   Data: 0.08033
Found entry with timestamp: 1787241559   Data: 0.08019
Found entry with timestamp: 1787241578   Data: 0.08025
Found entry with timestamp: 1787241597   Data: 0.08024
Found entry with timestamp: 1787241616   Data: 0.0802
Found entry with timestamp: 1787241636   Data: 0.07996
Found entry with timestamp: 1787241655   Data: 0.07976
Found entry with timestamp: 1787241675   Data: 0.07983
Found entry with timestamp: 1787241694   Data: 0.07987
Found entry with timestamp: 1787241713   Data: 0.07993
Found entry with timestamp: 1787241732   Data: 0.07991
Found entry with timestamp: 1787241751   Data: 0.07987
Found entry with timestamp: 1787241770   Data: 0.07989
Found entry with timestamp: 1787241789   Data: 0.07985
Found entry with timestamp: 1787241808   Data: 0.07986
Found entry with timestamp: 1787241827   Data: 0.07985
Found entry with timestamp: 1787241846   Data: 0.07981
Found entry

## Appendix



```
!pip install websocket-client

#https://pypi.org/project/websocket_client/
import websocket

def on_message(ws, message):
    print(message)

def on_error(ws, error):
    print(error)

def on_close(ws):
    print("### closed ###")

def on_open(ws):
    ws.send('{"type":"subscribe","symbol":"BINANCE:BTCUSDT"}')


websocket.enableTrace(False)
ws = websocket.WebSocketApp("wss://ws.finnhub.io?token=",
                          on_message = on_message,
                          on_error = on_error,
                          on_close = on_close)
ws.on_open = on_open
ws.run_forever()
```

